# Evaluate the Model on the Sample HDAs

In [ ]:
import sys, os
from pathlib import Path 

PROJECT_ROOT = Path("/Users/robertagarcia/Desktop/learning/bert_symptom_ner")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

sys.path

import torch
import json
from dataclasses import dataclass
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Local imports
from config import settings
from gcp_utils import download_from_gcs
from inference.v01.inference_utils import predict_word_level, word_labels_to_spans
from error_analysis.error_categorization import ErrorCategorizer
from error_analysis.error_taxonomy import BehaviouralExample
from sample_hdas import transcriptions_and_hdas

VERSION = os.environ["VERSION"]
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1" 
RUN_IDX = 0 # manually checked this in GCP bucket
INFERENCE_PIPELINE_VERSION = "v01"


# Move model to device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

LOCAL_MODEL_DIR = f"{PROJECT_ROOT}/{VERSION}/downloaded_models/dmis-lab/biobert-base-cased-v1.1/run_{RUN_IDX}"
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
model = model.to(device)
model.eval()
id2label = model.config.id2label

In [ ]:

#  HDA format → BehaviouralExample
from typing import Any


def hda_to_behavioural_example(hda: dict) -> BehaviouralExample:
    entities_with_labels = []
    for label, items in hda["entities"].items():
        for phrase, (start, end) in items.items():
            entities_with_labels.append({
                "ent": phrase,
                "start": start,
                "end": end,
                "label": label,   # "SYMPTOM_POS" / "SYMPTOM_NEG"
            })
    return BehaviouralExample(
        example=hda["template"],
        entities_with_labels=entities_with_labels,
    )

# Cell 3 — run evaluation (same pattern as _run_through_behavioural_set)
categorizer = ErrorCategorizer()
all_errors = {}
for i,hda in enumerate(transcriptions_and_hdas):
    if "entities" not in hda:
        continue
    example = hda_to_behavioural_example(hda)
    _, _, _, _, word_labels, word_offsets = predict_word_level(
        text=example.example, model=model, tokenizer=tokenizer,
        id2label=id2label, device=device,
    )
    spans = word_labels_to_spans(
        text=example.example, word_offsets=word_offsets, word_labels=word_labels,
    )
    all_errors[i] = categorizer._check_entity_detection(example=example, spans=spans)
print(dict[Any, int](categorizer.error_counts))
with open("all_errors_transcripts_and_hdas.json", "w") as f:
    json.dump(all_errors,f,indent= 1)

In [ ]:
for i, err in all_errors.items():
    n_present = len(err["present_errors"])
    n_missing = len(err["missing_entities"])
    n_fp = len(err["false_positives"])
    print(f"HDA {i}: present={n_present}, missing={n_missing}, FP={n_fp}, total={n_present+n_missing+n_fp}")